# GramNet v3 -- Real-World Retraining Pipeline
### Kaggle Notebook | T4 GPU Required

1. **Scrapes** thousands of real-world images
2. **Scrapes** AI-generated fake images for balance
3. **Lets you add** your own custom image directories
4. **Retrains** XGBoost using **top-8 eigenvalues**
5. **Evaluates** with confusion matrix, threshold sweep
6. **Saves** deployment-ready model files

In [ ]:
# CELL 0 -- Install dependencies
!pip install -q xgboost scikit-learn Pillow tqdm bing-image-downloader requests matplotlib seaborn

In [ ]:
# CELL 1 -- Verify GPU
import torch, platform
print(f'PyTorch : {torch.__version__}')
print(f'Python  : {platform.python_version()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print('GPU ready')
else:
    raise RuntimeError('No GPU! Enable T4 GPU in notebook settings.')

## Step 2 -- Scrape Real-World Images

In [ ]:
# CELL 2 -- Scrape real-world images from Bing
from bing_image_downloader import downloader
import os, shutil

SCRAPE_REAL_DIR = '/kaggle/working/scraped_real/'
SCRAPE_FAKE_DIR = '/kaggle/working/scraped_fake/'
os.makedirs(SCRAPE_REAL_DIR, exist_ok=True)
os.makedirs(SCRAPE_FAKE_DIR, exist_ok=True)

REAL_QUERIES = [
    'selfie portrait phone camera',
    'mirror selfie bathroom phone',
    'portrait mode bokeh phone photo',
    'whatsapp shared photo compressed',
    'instagram photo filter effect',
    'snapchat photo filter face',
    'facebook uploaded photo compressed',
    'photo taken with iphone indoor',
    'photo taken with samsung galaxy',
    'pixel phone camera photo',
    'birthday party celebration photos',
    'wedding ceremony reception photos',
    'graduation ceremony photos',
    'baby photo family newborn',
    'food photo restaurant phone',
    'pet photo cat dog phone camera',
    'group photo friends gathering',
    'travel photo landscape phone camera',
    'street photography mobile phone',
    'screenshot mobile phone conversation',
    'low quality phone photo blurry',
    'meme photo compressed low quality',
]

LIMIT_PER_QUERY = 300

print(f'Scraping {len(REAL_QUERIES)} categories x {LIMIT_PER_QUERY} images each')
print('=' * 60)

for i, query in enumerate(REAL_QUERIES):
    print(f'\n[{i+1}/{len(REAL_QUERIES)}] Scraping: "{query}"')
    try:
        downloader.download(query, limit=LIMIT_PER_QUERY, output_dir=SCRAPE_REAL_DIR,
                            adult_filter_off=False, force_replace=False, timeout=30)
    except Exception as e:
        print(f'  Warning: {e}')

total_real = 0
for root, dirs, files in os.walk(SCRAPE_REAL_DIR):
    total_real += len([f for f in files if f.lower().endswith(('.jpg','.jpeg','.png','.webp','.bmp'))])
print(f'\nTotal real images scraped: {total_real:,}')

In [ ]:
# CELL 3 -- Scrape AI-generated fake images
import requests
from PIL import Image
from io import BytesIO

FAKE_QUERIES = [
    'AI generated face portrait',
    'deepfake celebrity face',
    'midjourney generated portrait',
    'stable diffusion portrait realistic',
    'DALL-E generated face',
    'AI generated person full body',
    'artificial intelligence created face',
    'GAN generated face realistic',
]

print(f'Scraping {len(FAKE_QUERIES)} fake categories x {LIMIT_PER_QUERY} each')
for i, query in enumerate(FAKE_QUERIES):
    print(f'\n[{i+1}/{len(FAKE_QUERIES)}] Scraping: "{query}"')
    try:
        downloader.download(query, limit=LIMIT_PER_QUERY, output_dir=SCRAPE_FAKE_DIR,
                            adult_filter_off=False, force_replace=False, timeout=30)
    except Exception as e:
        print(f'  Warning: {e}')

# Download from thispersondoesnotexist.com
print('\nDownloading from thispersondoesnotexist.com...')
tpdne_dir = os.path.join(SCRAPE_FAKE_DIR, 'thispersondoesnotexist')
os.makedirs(tpdne_dir, exist_ok=True)
tpdne_count = 0
for idx in range(200):
    try:
        r = requests.get('https://thispersondoesnotexist.com/', timeout=10,
                         headers={'User-Agent': 'Mozilla/5.0'})
        if r.status_code == 200:
            img = Image.open(BytesIO(r.content))
            img.save(os.path.join(tpdne_dir, f'tpdne_{idx:04d}.jpg'), 'JPEG', quality=95)
            tpdne_count += 1
            if (idx+1) % 50 == 0: print(f'  Downloaded {tpdne_count} faces...')
    except Exception: pass

total_fake = 0
for root, dirs, files in os.walk(SCRAPE_FAKE_DIR):
    total_fake += len([f for f in files if f.lower().endswith(('.jpg','.jpeg','.png','.webp','.bmp'))])
print(f'\nTotal fake images scraped: {total_fake:,} (includes {tpdne_count} from TPDNE)')

## Step 3 -- Add Your Own Custom Images

In [ ]:
# CELL 4 -- Custom image management + heavy phone-simulation augmentation
import glob, random, io, shutil
import numpy as np
from PIL import Image, ImageFilter, ImageEnhance, ImageDraw

CUSTOM_REAL_DIR = '/kaggle/working/custom_real/'
CUSTOM_FAKE_DIR = '/kaggle/working/custom_fake/'
os.makedirs(CUSTOM_REAL_DIR, exist_ok=True)
os.makedirs(CUSTOM_FAKE_DIR, exist_ok=True)

IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def _find_images(directory):
    paths = []
    for ext in IMG_EXTENSIONS:
        paths.extend(glob.glob(os.path.join(directory, f'**/*{ext}'), recursive=True))
        paths.extend(glob.glob(os.path.join(directory, f'**/*{ext.upper()}'), recursive=True))
    return list(set(paths))

def add_custom_images(source_paths, count=None, label='real'):
    dest = CUSTOM_REAL_DIR if label == 'real' else CUSTOM_FAKE_DIR
    if isinstance(source_paths, str):
        source_paths = [source_paths]
    total_copied = 0
    for src in source_paths:
        if not os.path.isdir(src):
            print(f'  Warning: Directory not found: {src}')
            continue
        imgs = _find_images(src)
        if count is not None:
            random.shuffle(imgs)
            imgs = imgs[:count]
        for p in imgs:
            fname = f'{label}_{total_copied:05d}_{os.path.basename(p)}'
            shutil.copy2(p, os.path.join(dest, fname))
            total_copied += 1
        print(f'  Copied {len(imgs)} images from {src}')
    print(f'Total {label} images added: {total_copied} to {dest}')
    return total_copied

def augment_phone_heavy(img: Image.Image) -> Image.Image:
    if random.random() > 0.15:
        quality = random.choice([random.randint(15, 40), random.randint(40, 65), random.randint(65, 95)])
        buf = io.BytesIO()
        img.save(buf, 'JPEG', quality=quality)
        buf.seek(0)
        img = Image.open(buf).copy()
    if random.random() > 0.4:
        w, h = img.size
        scale = random.uniform(0.3, 0.85)
        small = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
        img = small.resize((w, h), Image.LANCZOS)
    if random.random() > 0.4:
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.7, 1.4))
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.7, 1.4))
        img = ImageEnhance.Color(img).enhance(random.uniform(0.6, 1.6))
    if random.random() > 0.7:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 2.0)))
    if random.random() > 0.6:
        img = ImageEnhance.Sharpness(img).enhance(random.uniform(0.5, 2.5))
    if random.random() > 0.75:
        w, h = img.size
        mask = Image.new('L', (w, h), 255)
        draw = ImageDraw.Draw(mask)
        cx, cy = w // 2, h // 2
        max_r = int(((cx**2 + cy**2) ** 0.5))
        for r in range(max_r, 0, -1):
            alpha = int(255 * (r / max_r) ** 2)
            draw.ellipse([cx - r, cy - r, cx + r, cy + r], fill=max(0, alpha))
        img = Image.composite(img, Image.new('RGB', (w, h), (0, 0, 0)), mask)
    if random.random() > 0.8:
        w, h = img.size
        blurred = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(3, 8)))
        mask = Image.new('L', (w, h), 0)
        draw = ImageDraw.Draw(mask)
        margin_x = int(w * random.uniform(0.15, 0.3))
        margin_y = int(h * random.uniform(0.1, 0.25))
        draw.ellipse([margin_x, margin_y, w - margin_x, h - margin_y], fill=255)
        mask = mask.filter(ImageFilter.GaussianBlur(radius=20))
        img = Image.composite(img, blurred, mask)
    if random.random() > 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    if random.random() > 0.7:
        arr = np.array(img).astype(np.float32)
        noise = np.random.normal(0, random.uniform(2, 12), arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        img = Image.fromarray(arr)
    return img

def augment_light(img: Image.Image) -> Image.Image:
    quality = random.randint(55, 92)
    buf = io.BytesIO()
    img.save(buf, 'JPEG', quality=quality)
    buf.seek(0)
    img = Image.open(buf).copy()
    if random.random() > 0.5:
        scale = random.uniform(0.85, 1.0)
        w, h = img.size
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
        img = img.resize((w, h), Image.LANCZOS)
    return img

print('Augmentation functions loaded.')

In [ ]:
# CELL 5 -- ADD YOUR CUSTOM IMAGES HERE
# add_custom_images('/kaggle/input/your-phone-photos/', count=500, label='real')
# add_custom_images('/kaggle/input/your-deepfake-dataset/', count=500, label='fake')

print('\nImage Summary:')
n_scraped_real = len(_find_images(SCRAPE_REAL_DIR))
n_scraped_fake = len(_find_images(SCRAPE_FAKE_DIR))
n_custom_real  = len(_find_images(CUSTOM_REAL_DIR))
n_custom_fake  = len(_find_images(CUSTOM_FAKE_DIR))
print(f'  Scraped real : {n_scraped_real:,}')
print(f'  Custom real  : {n_custom_real:,}')
print(f'  Scraped fake : {n_scraped_fake:,}')
print(f'  Custom fake  : {n_custom_fake:,}')
print(f'  -----------------')
print(f'  Total real   : {n_scraped_real + n_custom_real:,}')
print(f'  Total fake   : {n_scraped_fake + n_custom_fake:,}')

## Step 4 -- Configuration & Cache Loading

In [ ]:
# CELL 6 -- Extract cached features from previous training run
import zipfile, os
zip_file_path = '/kaggle/input/notebooks/mahmoudfathy06/el-mrady-aho/_output_.zip'
file_to_extract = 'GramNet_v3/cache/gram_features_v3_compact.pt'
extraction_destination = '/kaggle/working/'
try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extract(file_to_extract, path=extraction_destination)
        print(f'Extracted: {file_to_extract}')
except Exception as e:
    print(f'Failed to extract cache: {e}')

In [ ]:
# CELL 7 -- Configuration (TOP_K = 8 eigenvalues)
import os, json, glob, io, random
import numpy as np
import torch
import xgboost as xgb
from PIL import Image
from tqdm.auto import tqdm
from torchvision import transforms

for _p in ['/kaggle/working/GramNet_v3/cashe/gram_features_v3_compact.pt',
           '/kaggle/working/GramNet_v3/cache/gram_features_v3_compact.pt']:
    if os.path.exists(_p):
        CACHE_FILE = _p
        break
else:
    print('Warning: Cache not found. Feature arrays will be empty initially.')
    CACHE_FILE = None

CKPT_DIR    = '/kaggle/working/GramNet_v3/checkpoints/'
CONFIG_FILE = os.path.join(CKPT_DIR, 'config_detector_k8_inter_v3.json')
OLD_MODEL   = os.path.join(CKPT_DIR, 'xgb_detector_k8_inter_v3.json')
NORM_STATS  = os.path.join(CKPT_DIR, 'norm_stats_v3.pt')
SAVE_DIR    = CKPT_DIR
os.makedirs(SAVE_DIR, exist_ok=True)

NEW_REAL_DIRS = [SCRAPE_REAL_DIR, CUSTOM_REAL_DIR]
NEW_FAKE_DIRS = [SCRAPE_FAKE_DIR, CUSTOM_FAKE_DIR]

VGG_LAYER_INDICES  = [8, 22, 29]
VGG_CHANNELS       = [128, 512, 512]
TOP_K_EIGENVALUES  = 8    # Changed from 16
N_ENERGY_BANDS     = 4
N_SPECTRAL_STATS   = 4
N_INTER_LAYER      = 3
IMG_SZ             = 224
VGG_MEAN           = [0.485, 0.456, 0.406]
VGG_STD            = [0.229, 0.224, 0.225]
IMG_EXT            = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
DEVICE             = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH              = 32
random.seed(42); np.random.seed(42)

OLD_TOP_K = 16
def build_remap_indices():
    keep = []
    offset_old = 0
    for ch in VGG_CHANNELS:
        n_old = 2*ch + OLD_TOP_K + 1 + N_ENERGY_BANDS + N_SPECTRAL_STATS
        keep.extend(range(offset_old, offset_old + 2*ch))
        keep.extend(range(offset_old + 2*ch, offset_old + 2*ch + TOP_K_EIGENVALUES))
        keep.extend(range(offset_old + 2*ch + OLD_TOP_K, offset_old + n_old))
        offset_old += n_old
    return keep

REMAP_LAYER_INDICES = build_remap_indices()

def get_topk_positions_old():
    positions = []
    offset = 0
    for ch in VGG_CHANNELS:
        n_old = 2*ch + OLD_TOP_K + 1 + N_ENERGY_BANDS + N_SPECTRAL_STATS
        positions.append(list(range(offset + 2*ch, offset + 2*ch + TOP_K_EIGENVALUES)))
        offset += n_old
    return positions

TOPK_POS_OLD = get_topk_positions_old()
print(f'TOP_K_EIGENVALUES : {TOP_K_EIGENVALUES}')

In [ ]:
# CELL 8 -- Load cached features + remap from k=16 to k=8
if CACHE_FILE:
    cache = torch.load(CACHE_FILE, map_location='cpu', weights_only=False)
    X_orig_train = cache['X_train']
    y_orig_train = cache['y_train']
    X_orig_val   = cache['X_val']
    y_orig_val   = cache['y_val']
    feat_mean_old = cache['feat_mean']
    feat_std_old  = cache['feat_std']

    def remap_cached_features(X):
        X_layers = X[:, REMAP_LAYER_INDICES]
        topk_profiles = [X[:, pos] for pos in TOPK_POS_OLD]
        inter = []
        for i in range(len(topk_profiles)):
            for j in range(i + 1, len(topk_profiles)):
                cos = torch.nn.functional.cosine_similarity(topk_profiles[i], topk_profiles[j], dim=1).unsqueeze(1)
                inter.append(cos)
        X_inter = torch.cat(inter, dim=1)
        return torch.cat([X_layers, X_inter], dim=1)

    X_orig_train = remap_cached_features(X_orig_train)
    X_orig_val   = remap_cached_features(X_orig_val)
    feat_mean = torch.cat([feat_mean_old[REMAP_LAYER_INDICES], torch.zeros(N_INTER_LAYER)])
    feat_std  = torch.cat([feat_std_old[REMAP_LAYER_INDICES], torch.ones(N_INTER_LAYER)])
    all_train = torch.cat([X_orig_train], dim=0)
    feat_mean[-N_INTER_LAYER:] = all_train[:, -N_INTER_LAYER:].mean(dim=0)
    feat_std[-N_INTER_LAYER:]  = all_train[:, -N_INTER_LAYER:].std(dim=0)
    feat_std_safe = feat_std.clone()
    feat_std_safe[feat_std_safe < 1e-8] = 1.0
    FDIM_K8 = X_orig_train.shape[1]
    print(f'Remapped shape: {X_orig_train.shape}')
else:
    FDIM_K8 = 2339  # Calculated feature dimension for K=8
    X_orig_train = torch.empty(0, FDIM_K8)
    y_orig_train = torch.empty(0, dtype=torch.long)
    X_orig_val   = torch.empty(0, FDIM_K8)
    y_orig_val   = torch.empty(0, dtype=torch.long)
    feat_mean = torch.zeros(FDIM_K8)
    feat_std_safe = torch.ones(FDIM_K8)

In [ ]:
# CELL 9 -- VGG Gram Feature Extractor (k=8 eigenvalues)
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as Func

class VGGGramExtractorV3(nn.Module):
    def __init__(self, layer_indices=None, channels=None, top_k=None):
        super().__init__()
        self.layer_indices = layer_indices or VGG_LAYER_INDICES
        self.channels = channels or VGG_CHANNELS
        self.top_k = top_k or TOP_K_EIGENVALUES
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        self.features = vgg.features[:max(self.layer_indices) + 1]
        for p in self.features.parameters():
            p.requires_grad = False
        self.eval()

    @torch.no_grad()
    def extract_gram_features(self, x):
        all_layer_features, all_topk_profiles = [], []
        h = x
        for i, layer in enumerate(self.features):
            h = layer(h)
            if i not in self.layer_indices: continue
            B, C, H, W = h.shape
            N = H * W
            F = h.view(B, C, N)
            chan_mean = F.mean(dim=2)
            chan_std  = F.std(dim=2)
            G = torch.bmm(F, F.transpose(1, 2)) / N
            eigvals = torch.linalg.eigvalsh(G).flip(dims=[1]).clamp(min=0)
            total_energy = eigvals.sum(dim=1, keepdim=True) + 1e-10
            eigvals_norm = eigvals / total_energy
            k = min(self.top_k, C)
            topk_eigvals = eigvals_norm[:, :k]
            all_topk_profiles.append(topk_eigvals)
            log_eigvals = torch.log(eigvals + 1e-10)
            indices = torch.arange(1, C+1, device=x.device, dtype=torch.float32).unsqueeze(0)
            x_mean = indices.mean()
            y_mean = log_eigvals.mean(dim=1, keepdim=True)
            cov_xy = ((indices - x_mean) * (log_eigvals - y_mean)).mean(dim=1, keepdim=True)
            var_x  = ((indices - x_mean) ** 2).mean()
            slope  = cov_xy / (var_x + 1e-10)
            q_size = C // N_ENERGY_BANDS
            bands  = [eigvals_norm[:, q*q_size:(q+1)*q_size if q < N_ENERGY_BANDS-1 else C].sum(dim=1, keepdim=True)
                      for q in range(N_ENERGY_BANDS)]
            energy_bands = torch.cat(bands, dim=1)
            p = eigvals_norm.clamp(min=1e-10)
            entropy  = -(p * torch.log(p)).sum(dim=1, keepdim=True)
            eff_rank = torch.exp(entropy)
            cond     = torch.log(eigvals[:, 0:1] / (eigvals[:, -1:] + 1e-10) + 1)
            eig_mean = eigvals.mean(dim=1, keepdim=True)
            eig_std  = eigvals.std(dim=1, keepdim=True) + 1e-10
            kurtosis = ((eigvals - eig_mean) / eig_std).pow(4).mean(dim=1, keepdim=True) - 3.0
            layer_feat = torch.cat([chan_mean, chan_std, topk_eigvals, slope,
                                    energy_bands, entropy, eff_rank, cond, kurtosis], dim=1)
            all_layer_features.append(layer_feat)
        inter = [Func.cosine_similarity(all_topk_profiles[i], all_topk_profiles[j], dim=1).unsqueeze(1)
                 for i in range(len(all_topk_profiles))
                 for j in range(i+1, len(all_topk_profiles))]
        return torch.cat(all_layer_features + inter, dim=1)

gram_extractor = VGGGramExtractorV3().to(DEVICE)
_d = gram_extractor.extract_gram_features(torch.randn(1, 3, IMG_SZ, IMG_SZ, device=DEVICE))
print(f'Feature dim: {_d.shape[1]}')

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SZ, IMG_SZ)),
    transforms.ToTensor(),
    transforms.Normalize(VGG_MEAN, VGG_STD),
])

In [ ]:
# CELL 10 -- Extract features from all new images
def extract_from_dirs(directories, label, augment_fn=None, desc='Extracting'):
    all_paths = []
    for d in directories:
        if not os.path.isdir(d): continue
        for ext in IMG_EXT:
            all_paths.extend(glob.glob(os.path.join(d, f'**/*{ext}'), recursive=True))
            all_paths.extend(glob.glob(os.path.join(d, f'**/*{ext.upper()}'), recursive=True))
    all_paths = list(set(all_paths))
    if not all_paths:
        return torch.empty(0, FDIM_K8), torch.empty(0, dtype=torch.long)

    all_feats, failed = [], 0
    gram_extractor.eval()
    for i in tqdm(range(0, len(all_paths), BATCH), desc=desc):
        batch_paths = all_paths[i:i+BATCH]
        batch_imgs  = []
        for p in batch_paths:
            try:
                img = Image.open(p).convert('RGB')
                if augment_fn: img = augment_fn(img)
                batch_imgs.append(eval_transform(img))
            except Exception:
                failed += 1
        if not batch_imgs: continue
        imgs_t = torch.stack(batch_imgs).to(DEVICE)
        with torch.no_grad():
            feats = gram_extractor.extract_gram_features(imgs_t)
        all_feats.append(feats.cpu())

    if not all_feats:
        return torch.empty(0, FDIM_K8), torch.empty(0, dtype=torch.long)
    feats = torch.cat(all_feats, dim=0)
    labels = torch.zeros(feats.shape[0], dtype=torch.long) if label=='real' else torch.ones(feats.shape[0], dtype=torch.long)
    return feats, labels

print('Extracting Heavy Augmentation (Real)...')
X_real_heavy, y_real_heavy = extract_from_dirs(NEW_REAL_DIRS, 'real', augment_phone_heavy, 'Real (Heavy)')
print('Extracting Light Augmentation (Real)...')
X_real_light, y_real_light = extract_from_dirs(NEW_REAL_DIRS, 'real', augment_light, 'Real (Light)')
print('Extracting Clean (Real)...')
X_real_clean, y_real_clean = extract_from_dirs(NEW_REAL_DIRS, 'real', None, 'Real (Clean)')

print('Extracting Light Augmentation (Fake)...')
X_fake_light, y_fake_light = extract_from_dirs(NEW_FAKE_DIRS, 'fake', augment_light, 'Fake (Light)')
print('Extracting Clean (Fake)...')
X_fake_clean, y_fake_clean = extract_from_dirs(NEW_FAKE_DIRS, 'fake', None, 'Fake (Clean)')

X_new = torch.cat([X_real_heavy, X_real_light, X_real_clean, X_fake_light, X_fake_clean], dim=0)
y_new = torch.cat([y_real_heavy, y_real_light, y_real_clean, y_fake_light, y_fake_clean], dim=0)

In [ ]:
# CELL 11 -- Combine & Normalize Data
from sklearn.model_selection import train_test_split
if X_new.shape[0] > 0:
    X_new_train, X_new_val, y_new_train, y_new_val = train_test_split(
        X_new.numpy(), y_new.numpy(), test_size=0.2, random_state=42, stratify=y_new.numpy()
    )
    X_new_train = torch.from_numpy(X_new_train)
    X_new_val = torch.from_numpy(X_new_val)
    y_new_train = torch.from_numpy(y_new_train)
    y_new_val = torch.from_numpy(y_new_val)
else:
    X_new_train = torch.empty(0, FDIM_K8)
    X_new_val = torch.empty(0, FDIM_K8)
    y_new_train = torch.empty(0, dtype=torch.long)
    y_new_val = torch.empty(0, dtype=torch.long)

X_train_comb = torch.cat([X_orig_train, X_new_train], dim=0)
y_train_comb = torch.cat([y_orig_train, y_new_train], dim=0)
X_val_comb   = torch.cat([X_orig_val, X_new_val], dim=0)
y_val_comb   = torch.cat([y_orig_val, y_new_val], dim=0)

# Recompute global mean/std from training set only
feat_mean = X_train_comb.mean(dim=0)
feat_std  = X_train_comb.std(dim=0)
feat_std_safe = feat_std.clone()
feat_std_safe[feat_std_safe < 1e-8] = 1.0

X_train_norm = (X_train_comb - feat_mean) / feat_std_safe
X_val_norm   = (X_val_comb - feat_mean) / feat_std_safe

print(f'Final Training Data: {X_train_norm.shape[0]:,} samples')
print(f'Final Valid Data:    {X_val_norm.shape[0]:,} samples')

## Step 6 -- Train XGBoost Model

In [ ]:
# CELL 12 -- Train XGBoost
num_fake = (y_train_comb == 1).sum().item()
num_real = (y_train_comb == 0).sum().item()
scale_pos = num_real / max(num_fake, 1)

clf = xgb.XGBClassifier(
    n_estimators=600,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    objective='binary:logistic',
    tree_method='hist',
    device='cuda',
    scale_pos_weight=scale_pos,
    random_state=42,
    eval_metric=['auc', 'logloss']
)
print('Training... (this might take a while)')
clf.fit(
    X_train_norm.numpy(), y_train_comb.numpy(),
    eval_set=[(X_train_norm.numpy(), y_train_comb.numpy()), (X_val_norm.numpy(), y_val_comb.numpy())],
    verbose=50
)

## Step 7 -- Evaluation & Threshold Analysis

In [ ]:
# CELL 13 -- Confusion Matrix & Metrics
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, f1_score

y_val_pred_prob = clf.predict_proba(X_val_norm.numpy())[:, 1]
y_val_pred_05 = (y_val_pred_prob > 0.5).astype(int)
print('\n--- Classification Report (Threshold=0.5) ---')
print(classification_report(y_val_comb.numpy(), y_val_pred_05, target_names=['Real (0)', 'Fake (1)']))

precisions, recalls, thresholds = precision_recall_curve(y_val_comb.numpy(), y_val_pred_prob)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
best_thresh = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
y_val_pred_best = (y_val_pred_prob > best_thresh).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(confusion_matrix(y_val_comb.numpy(), y_val_pred_05), annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix (Thresh = 0.5)')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

sns.heatmap(confusion_matrix(y_val_comb.numpy(), y_val_pred_best), annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title(f'Confusion Matrix (Best Thresh = {best_thresh:.3f})')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150)
plt.show()

## Step 8 -- Save Deployment Files

In [ ]:
# CELL 14 -- Save XGBoost and Configs
NEW_MODEL_PATH = os.path.join(SAVE_DIR, 'xgb_detector_k8_retrained.json')
NEW_CONFIG_PATH = os.path.join(SAVE_DIR, 'config_detector_k8_retrained.json')
NEW_STATS_PATH = os.path.join(SAVE_DIR, 'norm_stats_v3_retrained.pt')
NEW_THRESHOLD_PATH = os.path.join(SAVE_DIR, 'threshold_config.json')

clf.save_model(NEW_MODEL_PATH)
torch.save({'feat_mean': feat_mean, 'feat_std': feat_std_safe}, NEW_STATS_PATH)

new_config = {
    "input_dim": FDIM_K8,
    "layer_indices": VGG_LAYER_INDICES,
    "channels": VGG_CHANNELS,
    "top_k_eigenvalues": TOP_K_EIGENVALUES,
    "n_energy_bands": N_ENERGY_BANDS,
    "n_spectral_stats": N_SPECTRAL_STATS,
    "use_inter_layer": True,
    "inter_layer_dim": N_INTER_LAYER,
    "image_size": IMG_SZ
}
with open(NEW_CONFIG_PATH, 'w') as f:
    json.dump(new_config, f, indent=4)

threshold_config = {
    "threshold": float(best_thresh),
    "threshold_mode": "auto",
    "notes": "Set mode to 'manual' to override the optimal threshold"
}
with open(NEW_THRESHOLD_PATH, 'w') as f:
    json.dump(threshold_config, f, indent=4)

print('Model saved successfully!')